# Regression Analysis with Python

In [ ]:
# Loading data
import pandas as pd
import janitor
pd.set_option("display.max_columns", None)
data_path = "../../data/raw/WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv"
df = pd.read_csv(data_path)
df.head()

In [ ]:
# rename columns to remove spaces and make them to sneak case
df = df.clean_names()
df.head()

In [ ]:
df['engaged'] = df['response'].apply(lambda x: 0 if x == 'No' else 1)

In [ ]:
engagement_rate_df = pd.DataFrame(df.groupby('engaged').count()['response'] / df.shape[0] * 100.0)
engagement_rate_df

In [ ]:
engagement_rate_df.T

# Sales Channels

In [ ]:
engagement_by_sales_channel_df = pd.pivot_table(df,
                                                values='response',
                                                index='sales_channel',
                                                columns='engaged',
                                                aggfunc=len
                                                ).fillna(0.0)
engagement_by_sales_channel_df.columns = ['Not Engaged', 'Engaged']
engagement_by_sales_channel_df


In [ ]:
engagement_by_sales_channel_df.reset_index(inplace=True)
engagement_by_sales_channel_df.columns

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly import colors

plot_df = engagement_by_sales_channel_df

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Not Engaged (0)', 'Engaged (1)']
)

for i, col_name in enumerate(['Not Engaged', 'Engaged'], start=1):
    fig.add_trace(
        go.Pie(
            labels=plot_df.index,
            values=plot_df[col_name],
            name=col_name,
            textinfo='percent+label',
            marker=dict(colors=colors.qualitative.Pastel)
        ),
        row=1, col=i
    )

fig.update_layout(
    title='Sales Channel Distribution by Engagement',
    template='ggplot2'
)
fig.show()

# Total Claim Amount by Engagement

In [ ]:
fig = go.Figure()

for engaged_value in sorted(df["engaged"].unique()):
    fig.add_trace(
        go.Box(
            y=df.loc[df["engaged"] == engaged_value, "total_claim_amount"],
            name=str(engaged_value),
            boxpoints=False  # similar to showfliers=False
        )
    )

fig.update_layout(
    title="Total Claim Amount Distributions by Engagements",
    xaxis_title="Engaged",
    yaxis_title="Total Claim Amount",
    template="ggplot2"
)

fig.show()

In [ ]:
fig = go.Figure()

for engaged_value in sorted(df["engaged"].unique()):
    fig.add_trace(
        go.Box(
            y=df.loc[df["engaged"] == engaged_value, "total_claim_amount"],
            name=str(engaged_value),
            boxpoints='outliers'
        )
    )

fig.update_layout(
    title="Total Claim Amount Distributions by Engagements",
    xaxis_title="Engaged",
    yaxis_title="Total Claim Amount",
    template="ggplot2"
)

fig.show()

# Logistic Regression Analysis

## continuous variables

In [ ]:
df['income'].dtype

In [ ]:
df['customer_lifetime_value'].dtype

In [ ]:
df.describe()

In [ ]:
# Create list of continuous variables
continuous_vars = ['customer_lifetime_value',
                   'income', 'monthly_premium_auto',
                   'months_since_last_claim',
                   'months_since_policy_inception',
                   'number_of_open_complaints',
                   'number_of_policies',
                   'total_claim_amount']


In [ ]:
import statsmodels.api as sm

X = sm.add_constant(df[continuous_vars])
y = df['engaged']

logit = sm.Logit(y, X).fit()
logit.summary()

## Categorical variables

In [ ]:
# using .factorize() to convert categorical variables into numeric codes
# Example
gender_values, gender_labels = df['gender'].factorize()

In [ ]:
gender_values

In [ ]:
gender_labels

In [ ]:
# Create categorical for ordinal variable
# 0: High School or Below
# 1: Bachelor
# 2: College
# 3: Master
# 4: Doctor
categories = pd.Categorical(
    df['education'],
    categories=['High School or Below', 'Bachelor', 'College', 'Master', 'Doctor'],
    ordered=True
)


In [ ]:
categories.codes

In [ ]:
# Create new column in the dataframe for encoded data as gender_factorized and education_factorized
df['gender_factorized'] = gender_values
df['education_factorized'] = categories.codes
df.head()

In [ ]:
# Create Logistic Regression model

X = sm.add_constant(df[['gender_factorized', 'education_factorized']])
y = df['engaged']
logit = sm.Logit(y, X).fit()
logit.summary()

# Combining Continuous and Categorical Variables in Logistic Regression

In [ ]:
# Create list of variables to include in the model
model_vars = ['customer_lifetime_value', 'income', 'monthly_premium_auto',
              'months_since_last_claim', 'months_since_policy_inception',
              'number_of_open_complaints', 'number_of_policies',
              'total_claim_amount','gender_factorized', 'education_factorized']

X = sm.add_constant(df[model_vars])
y = df['engaged']
logit = sm.Logit(y, X).fit()
logit.summary()